# Project 09: Forward-Backward Algorithm

This notebook implements the Hidden Markov Model (HMM) Forward-Backward algorithm. 

The forward and backward matrices are computed in log space to prevent underflow, the final forward-backward (posterior) matrix is converted back to normal space after calculations. Instead of creating an additional column at the initial index of the backward matrix to account for the initial state distribution and emission probability of the first observation, this implementation maintains a dictionary strictly for the observations and incorporates the initial and emission probabilities separately during the calculation of the total backward probability. 

## Main Algorithm

In [6]:
import math
import numpy as np
from viterbi_hmm import *
from pprint import pprint


class ForwardBackward(HMM):
    """
    Implementation of the Forward-Backward algorithm.
    """

    def forward(self, observation_sequence):
        """ Computes forward probabilities for a given observation sequence. """

        n = len(observation_sequence)
        if n == 0:
            return [], []
        # Initialize Forward Matrix
        forward_matrix = {s: [float("-inf")] * n for s in self.states}

        # INITIALIZATION (t = 0)
        for s in self.states:
            # Initial probability + Emission probability
            forward_matrix[s][0] = self.log_initial[s] + self.log_emission[s].get(observation_sequence[0], float("-inf"))

        # ITERATION (t = 1 to n-1)
        for t in range(1, n):
            obs = observation_sequence[t]
            for current_state in self.states:

                # Initialize an empty list to store the values that need to be added for the current observation
                values = []

                for prev_state in self.states:

                    # Probability in matrix at t-1 for previous state + 
                    # transition probability of previous state to current state + 
                    # emission probability of current observation at current state
                    values.append(forward_matrix[prev_state][t-1] + 
                                  self.log_transition[prev_state][current_state] + 
                                  self.log_emission[current_state].get(obs, float("-inf")))
                
                # log-sum-exp to add probabilities safely since the probabilities are in log-space
                forward_matrix[current_state][t] = np.logaddexp.reduce(values)

        # Add final probabilities (using log-sum-exp) to get total log probability
        final_values = [forward_matrix[s][n-1] for s in self.states]
        total_forward_prob = np.logaddexp.reduce(final_values)

        return forward_matrix, total_forward_prob
    

    def backward(self, observation_sequence):
        """ Computes the backwards probabilities of a given observation sequence. """

        n = len(observation_sequence)
        if n == 0:
            return [], []
        
        # Initialize Backward Matrix
        backward_matrix = {s: [float("-inf")] * n for s in self.states}

        # INITIALIZATION (t = n-1)
        for s in self.states:
            # Initialize to 0.0 since log(1) = 0
            backward_matrix[s][n-1] = 0.0

        # ITERATION (t = n-2 to 0)
        for t in range(n-2, -1, -1):
            next_obs = observation_sequence[t+1]
            for current_state in self.states:

                # Initialize an empty list to store the values that need to be added for the current observation
                values = []

                for next_state in self.states:

                    # Probability in matrix at t+1 for next state + 
                    # transition probability from current state to next state + 
                    # emission probability of next observation at next state
                    values.append(backward_matrix[next_state][t+1] + 
                                  self.log_transition[current_state][next_state] + 
                                  self.log_emission[next_state].get(next_obs, float("-inf")))
                    
                # log-sum-exp to add probabilities safely since the probabilities are in log-space
                backward_matrix[current_state][t] = np.logaddexp.reduce(values)
        
        # Add initial probability and emission probability for observation[0] to get final probabilities for each state
        final_values = [(backward_matrix[s][0] + 
                         self.log_emission[s].get(observation_sequence[0], float("-inf")) + 
                         self.log_initial[s]) for s in self.states]
        
        # Add final probabilities (using log-sum-exp) to get total log probability
        total_backward_prob = np.logaddexp.reduce(final_values)

        return backward_matrix, total_backward_prob
    

    def run(self, observation_sequence):
        
        # Run forward and backward algorithm
        forward_matrix, total_forward_prob = self.forward(observation_sequence)
        backward_matrix, total_backward_prob = self.backward(observation_sequence)

        n = len(observation_sequence)
        # Initialize a matrix to hold probabilities for each observation at each state
        final_prob_matrix = {s: [0.0] * n for s in self.states}

        # Calculate the probability for each observation at each state using the forward and backward matrices
        for t in range(n):
            for s in self.states:
                # Normalizing the final result
                final_prob_matrix[s][t] = math.exp(forward_matrix[s][t] + backward_matrix[s][t] - total_forward_prob)

        return forward_matrix, total_forward_prob, backward_matrix, total_backward_prob, final_prob_matrix
    
print("Successfully initialized HMM and ForwardBackward classes.")

Successfully initialized HMM and ForwardBackward classes.


## Test 1: Exon-Intron Forward-Backward Probability Calculation

This was the example provided as part of the assignment.

In [7]:
# Example observation sequence
obs = "ATGCAA"

states = ['E', 'I']

# Example initial probabilities (probability of starting in each state: E := Exon, I := Intron)
init_probs = {
    "E": 0.6,
    "I": 0.4
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "E": {"E": 0.8, "I": 0.2},
    "I": {"E": 0.3, "I": 0.7}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "E": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3},
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1}
}


fb_model = ForwardBackward(states, init_probs, trans_probs, emit_probs)
viterbi_model = Viterbi(states, init_probs, trans_probs, emit_probs)

# Execute the Forward Backward Algorithm
f_matrix, total_forward_prob, b_matrix, total_backward_prob, forward_backward_matrix = fb_model.run(list(obs))
viterbi_path = viterbi_model.run(list(obs))

# Convert dictionary values from NumPy scalars to Python floats for cleaner output
forward_matrix = {k: [float(v) for v in vals] for k, vals in f_matrix.items()}
backward_matrix = {k: [float(v) for v in vals] for k, vals in b_matrix.items()}

# Display Results
print(f"Input DNA Sequence: {obs}")
print("-" * 50)
print(f"Sequence: {'  '.join(obs)}")
print("-" * 50)
print(f"Optimal path (from viterbi analysis): {'  '.join(viterbi_path)}")
print("-" * 50)
print("Forward matrix:")
pprint(forward_matrix)
print(f"Total probability (forward): {total_forward_prob}")
print("-" * 50)
print("Backward_matrix:")
pprint(backward_matrix)
print(f"Total probability (backward): {total_backward_prob}")
print("-" * 50)
print("Final matrix:")
pprint(forward_backward_matrix)

Input DNA Sequence: ATGCAA
--------------------------------------------------
Sequence: A  T  G  C  A  A
--------------------------------------------------
Optimal path (from viterbi analysis): E  E  E  E  E  E
--------------------------------------------------
Forward matrix:
{'E': [-1.7147984280919268,
       -3.0618720760585365,
       -4.844443119232185,
       -6.44296521678929,
       -7.455610099029687,
       -8.769744132435449],
 'I': [-3.2188758248682006,
       -5.051457288616511,
       -5.196483060666769,
       -6.12850379937006,
       -8.598281779165047,
       -10.617926032715248]}
Total probability (forward): -8.623461496768252
--------------------------------------------------
Backward_matrix:
{'E': [-7.054945704343522,
       -5.74081167093776,
       -4.322701680589199,
       -2.724179583032094,
       -1.3470736479666094,
       0.0],
 'I': [-7.399050207847047,
       -5.379405954296847,
       -4.29592233562175,
       -3.363901596918459,
       -1.8325814637483

## Test 2: High-Contrast Emission Forward-Backward Test

In this test case, we evaluate how the Forward-Backward Algorithm responds to sequences with significant differences in emission probabilities between states. We assign high G/C probabilities to introns and high A/T probabilities to exons and allow more flexible state transitions.

In [8]:
# Example observation sequence
obs = "ATGCAA"

states = ['E', 'I']

# Example initial probabilities (probability of starting in each state: E := Exon, I := Intron)
init_probs = {
    "E": 0.5,
    "I": 0.5
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "E": {"E": 0.6, "I": 0.4},
    "I": {"E": 0.4, "I": 0.6}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "E": {"A": 0.45, "C": 0.05, "G": 0.05, "T": 0.45},
    "I": {"A": 0.05, "C": 0.45, "G": 0.45, "T": 0.05}
}


fb_model = ForwardBackward(states, init_probs, trans_probs, emit_probs)
viterbi_model = Viterbi(states, init_probs, trans_probs, emit_probs)

# Execute the Forward Backward Algorithm
f_matrix, total_forward_prob, b_matrix, total_backward_prob, forward_backward_matrix = fb_model.run(list(obs))
viterbi_path = viterbi_model.run(list(obs))

# Convert dictionary values from NumPy scalars to Python floats for cleaner output
forward_matrix = {k: [float(v) for v in vals] for k, vals in f_matrix.items()}
backward_matrix = {k: [float(v) for v in vals] for k, vals in b_matrix.items()}

# Display Results
print(f"Input DNA Sequence: {obs}")
print("-" * 50)
print(f"Sequence: {'  '.join(obs)}")
print("-" * 50)
print(f"Optimal path (from viterbi analysis): {'  '.join(viterbi_path)}")
print("-" * 50)
print("Forward matrix:")
pprint(forward_matrix)
print(f"Total probability (forward): {total_forward_prob}")
print("-" * 50)
print("Backward_matrix:")
pprint(backward_matrix)
print(f"Total probability (backward): {total_backward_prob}")
print("-" * 50)
print("Final matrix:")
pprint(forward_backward_matrix)

Input DNA Sequence: ATGCAA
--------------------------------------------------
Sequence: A  T  G  C  A  A
--------------------------------------------------
Optimal path (from viterbi analysis): E  E  I  I  E  E
--------------------------------------------------
Forward matrix:
{'E': [-1.491654876777717,
       -2.7295292327793343,
       -6.183836439755438,
       -8.03130083419546,
       -7.13804146579263,
       -8.34817298794737],
 'I': [-3.6888794541139363,
       -5.249527202378604,
       -4.3303834015220435,
       -5.540357081121116,
       -8.993164592441765,
       -10.839276520538155]}
Total probability (forward): -8.268605602585021
--------------------------------------------------
Backward_matrix:
{'E': [-6.8565181111696525,
       -5.646386589014912,
       -4.342421380081523,
       -2.4949569856415015,
       -1.2378743560016172,
       0.0],
 'I': [-7.150397066424218,
       -5.3042851383278276,
       -4.0487022043434,
       -2.8387285247443264,
       -1.5606477482